# RAG Chain — Groq LLM + Citation-Aware Retrieval

**Goal:** Wire the `HybridQdrantRetriever` to `llama-3.3-70b-versatile` (via Groq) and run end-to-end question answering with inline citations and resolved source URLs.

**Pipeline overview:**

```
question (str)
    │
    ▼
HybridQdrantRetriever          ← BGE-M3 dense + keyword sparse + RRF fusion
    │  top_k Document objects
    ▼
_format_docs()                 ← numbered [N] citation blocks
    │  context str
    ▼
ChatPromptTemplate             ← system: citation rules  |  human: context + question
    │
    ▼
ChatGroq (llama-3.3-70b)       ← temperature=0, max_tokens=1024
    │  answer str with [N] markers
    ▼
_extract_sources()             ← resolves markers → SourceRef list
    │
    ▼
RAGResult(answer, sources, context_docs)
```

## 1. Environment Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /home/dmitry/Projects/DataScience/rag-techdoc-assistant


In [2]:
import logging
import os

from dotenv import load_dotenv
from src.vectorstore import show_results

load_dotenv(PROJECT_ROOT / ".env")

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s  %(name)-25s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

log = logging.getLogger("notebook")

## 2. Configuration

In [3]:
COLLECTION_NAME = "pytorch_docs"
TOP_K           = 6      # chunks retrieved per query
MAX_TOKENS      = 1024   # max answer length
TEMPERATURE     = 0.0    # deterministic answers

QDRANT_URL = os.environ["QDRANT_URL"]
QDRANT_KEY = os.environ["QDRANT_API_KEY"]
GROQ_KEY   = os.environ["GROQ_API_KEY"]

print(f"Collection : {COLLECTION_NAME}")
print(f"Qdrant URL : {QDRANT_URL}")
print(f"Top-K      : {TOP_K}")

Collection : pytorch_docs
Qdrant URL : https://47c266d8-8135-4e60-9e8b-6fd120ff239b.europe-west3-0.gcp.cloud.qdrant.io
Top-K      : 6


## 3. Instantiate the Retriever

Reconnect to the populated Qdrant collection and wrap it in the `HybridQdrantRetriever`.

In [4]:
from qdrant_client import QdrantClient

from src.embedding import BGEM3Embedder
from src.vectorstore import QdrantDocStore

qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_KEY)

# BGE-M3 is used at query time for the dense leg.
# batch_size=1 is fine for single-query retrieval.
embedder = BGEM3Embedder(batch_size=1)

store = QdrantDocStore(
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    embedder=embedder,
)

retriever = store.as_retriever(top_k=TOP_K)

info = store.collection_info()
print(f"Collection '{info['name']}' — {info['points_count']:,} points, status: {info['status']}")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Collection 'pytorch_docs' — 8,358 points, status: green


## 4. Build the RAG Chain

In [5]:
from src.rag import build_rag_chain, print_result

chain = build_rag_chain(
    retriever=retriever,
    groq_api_key=GROQ_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

print("Chain ready:", chain)

Chain ready: first=RunnableLambda(retrieve_and_pack) middle=[RunnableLambda(build_prompt_input), RunnableLambda(llm_step)] last=RunnableLambda(pack_result)


## 5. Single-Query Demo

In [32]:
# question = "How does `torch.autograd.grad` differ from calling .backward()?"
question = "What is torch.cos?"

result = chain.invoke(question)
print_result(result)

Inference Embeddings: 100%|████████████████████████████| 1/1 [00:00<00:00, 91.61it/s]


There is no information about `torch.cos` in the provided context passages [1][2][3][4][5][6].

Sources
----------------------------------------
  [1] torch.package
       https://docs.pytorch.org/docs/stable/package.html#torch.package.PackageImporter.load_binary
  [2] torch.cov
       https://docs.pytorch.org/docs/stable/generated/torch.cov.html#torch.cov
  [3] torch.cdist
       https://docs.pytorch.org/docs/stable/generated/torch.cdist.html#torch.cdist
  [4] torch.package.Directory
       https://docs.pytorch.org/docs/stable/package.html#torch.package.Directory
  [5] torch.compile Troubleshooting
       https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/torch.compiler_troubleshooting.html#changing-the-cache-size-limit
  [6] torch.std
       https://docs.pytorch.org/docs/stable/generated/torch.std.html#torch.std


### 5a. Inspect the retrieved context

The raw documents that formed the context are available on `result.context_docs`.

In [15]:
print(f"Retrieved {len(result.context_docs)} chunks:\n")
for i, doc in enumerate(result.context_docs, 1):
    m = doc.metadata
    print(f"  [{i}] kind={m.get('kind'):<12}  score={m.get('score', 0):.4f}  "
          f"symbol={m.get('symbol') or '—'}")
    print(f"       {m.get('citation_url')}")

Retrieved 6 chunks:

  [1] kind=function      score=0.5000  symbol=torch.signal.windows.cosine
       https://docs.pytorch.org/docs/stable/generated/torch.signal.windows.cosine.html#torch.signal.windows.cosine
  [2] kind=heading       score=0.5000  symbol=—
       https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/torch.compiler_faq.html#how-are-you-speeding-up-my-code
  [3] kind=heading       score=0.3333  symbol=—
       https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/torch.compiler_faq.html#can-i-execute-numpy-code-on-cuda-and-compute-gradients-via-torch-compile
  [4] kind=function      score=0.3333  symbol=torch.cov
       https://docs.pytorch.org/docs/stable/generated/torch.cov.html#torch.cov
  [5] kind=heading       score=0.2500  symbol=—
       https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/torch.compiler_faq.html#compiling-torch-func-grad-with-torch-compile
  [6] kind=function      score=0.2500  symbol=torch.signal.windows.general_cosi

## 5b. Play around

In [35]:
show_results(store, "What is torch.cos ?") # with a space before the questionmark

Inference Embeddings: 100%|███████████████████████████| 1/1 [00:00<00:00, 100.43it/s]


Query: 'What is torch.cos ?'

#    Score  Kind         Symbol / Title                      Citation
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
1   0.5000  function     torch.cos                           https://docs.pytorch.org/docs/stable/generated/torch.cos.html#torch.cos
2   0.5000  function     torch.cov                           https://docs.pytorch.org/docs/stable/generated/torch.cov.html#torch.cov
3   0.3333  function     torch.cdist                         https://docs.pytorch.org/docs/stable/generated/torch.cdist.html#torch.cdist
4   0.3333  object       torch.package                       https://docs.pytorch.org/docs/stable/package.html#torch.package.PackageImporter.load_binary



## 6. Streaming Demo

For interactive applications, build the chain with `streaming=True` to receive tokens as they are generated.

> **Note:** The streaming chain returns raw token strings; `RAGResult` source extraction is not available in this mode.

In [45]:
stream_chain = build_rag_chain(
    retriever=retriever,
    groq_api_key=GROQ_KEY,
    streaming=True,
)

print("Streaming answer:\n")
for token in stream_chain.stream("give an example of torch.autocast"):
    print(token, end="", flush=True)
print()  # final newline

Streaming answer:



Inference Embeddings: 100%|████████████████████████████| 1/1 [00:00<00:00, 94.28it/s]


An example of `torch.autocast` can be found by looking at `torch.cuda.amp.autocast` [5], which is similar to `torch.autocast`. Here is an example:
```python
with torch.cuda.amp.autocast():
    # code to run with autocast
```
However, since `torch.cuda.amp.autocast(args...)` is deprecated [5], the recommended way is to use `torch.amp.autocast("cuda", args...)` instead. Unfortunately, the context does not contain enough information to provide a complete example of `torch.autocast` [5].


## 7. Filtered Retrieval

Scope retrieval to API-reference chunks only (exclude `heading` chunks) for symbol-specific questions.

In [ ]:
from qdrant_client import models as qdrant_models

# Restrict to function / class / method / attribute chunks.
api_filter = qdrant_models.Filter(
    must=[
        qdrant_models.FieldCondition(
            key="kind",
            match=qdrant_models.MatchExcept(**{"except": ["heading"]}),
        )
    ]
)

api_retriever = store.as_retriever(top_k=TOP_K, filter_=api_filter)
api_chain     = build_rag_chain(retriever=api_retriever, groq_api_key=GROQ_KEY)

result = api_chain.invoke("What parameters does torch.nn.MultiheadAttention accept?")
print_result(result)

## 8. Batch Evaluation

Run a small set of evaluation questions and collect answers + cited URLs for offline review.

In [ ]:
eval_questions = [
    "How do I move a tensor to GPU?",
    "What is the difference between torch.Tensor and torch.tensor?",
    "How does gradient checkpointing reduce memory usage?",
    "What does torch.no_grad() do and when should I use it?",
    "How do I save and load a model checkpoint?",
]

eval_results = []
for q in eval_questions:
    r = chain.invoke(q)
    eval_results.append({"question": q, "result": r})
    print(f"Q: {q}")
    print(f"A: {r.answer[:200]}{'…' if len(r.answer) > 200 else ''}")
    print(f"   Sources: {[s.url for s in r.sources]}")
    print()

## 9. Prompt Inspection

Print the exact prompt sent to the model for a given question

In [49]:
from src.rag.chain import _PROMPT, _format_docs

debug_question = "What is torch.autocast"
debug_docs     = retriever.invoke(debug_question)
debug_context  = _format_docs(debug_docs)

rendered = _PROMPT.format_messages(
    context=debug_context,
    question=debug_question,
)

for msg in rendered:
    role = msg.__class__.__name__.replace("Message", "").upper()
    print(f"{'─'*60}\n[{role}]\n{msg.content}")

Inference Embeddings: 100%|████████████████████████████| 1/1 [00:00<00:00, 95.51it/s]


────────────────────────────────────────────────────────────
[SYSTEM]
You are a precise technical assistant for the PyTorch documentation.

Answer the user's question using ONLY the context passages provided below.
Each passage is prefixed with a citation marker [N].

Rules:
- Cite every factual claim with its marker, e.g. "torch.Tensor is the central data structure [1]."
- A single sentence may carry multiple markers if supported by several passages, e.g. "[1][3]".
- If the context does not contain enough information to answer, say so explicitly — do not hallucinate.
- Prefer concise, technically accurate prose over bullet lists unless a list is clearly the best format.
- Preserve exact PyTorch symbol names, parameter names, and version notes as they appear in the context.

────────────────────────────────────────────────────────────
[HUMAN]
## Context

[1] **torch.cuda.amp.autocast** (https://docs.pytorch.org/docs/stable/amp.html#torch.cuda.amp.autocast)
```python
classtorch.cuda.amp